# 02 — Sauvegarder, charger et suivre l'entraînement avec TensorBoard

Entraîner un modèle prend du temps. Ce notebook montre comment :
- Sauvegarder des checkpoints réguliers
- Reprendre l'entraînement là où on s'est arrêté
- Visualiser la progression avec TensorBoard

Source : https://pythonprogramming.net/saving-and-loading-reinforcement-learning-stable-baselines-3-tutorial/

## Pourquoi sauvegarder ?

Pour aller de 10K à 1M de timesteps, on ne peut pas tout faire d'un coup :
- L'ordinateur peut s'éteindre
- On veut comparer plusieurs niveaux d'entraînement
- On veut reprendre sans tout recommencer

La stratégie : entraîner par **tranches** (ex: 10K steps) et sauvegarder à chaque tranche.

## 1. Préparation des dossiers

In [ ]:
import os
import gymnasium as gym
from stable_baselines3 import PPO

# Dossiers pour les modèles et les logs TensorBoard
models_dir = "models/PPO"
logs_dir = "logs"

os.makedirs(models_dir, exist_ok=True)
os.makedirs(logs_dir, exist_ok=True)

print(f"Dossier modèles : {models_dir}")
print(f"Dossier logs    : {logs_dir}")

## 2. Entraînement avec checkpoints réguliers

Idée : boucle infinie avec des tranches de `TIMESTEPS` steps.
À chaque tranche, on sauvegarde le modèle avec le total cumulé comme nom de fichier.

In [ ]:
env = gym.make('LunarLander-v3')

# tensorboard_log indique où écrire les métriques
model = PPO('MlpPolicy', env, verbose=1, tensorboard_log=logs_dir)

TIMESTEPS = 10_000  # taille d'une tranche d'entraînement
N_ITERATIONS = 5    # nombre de tranches (= 50K steps au total)

for i in range(1, N_ITERATIONS + 1):
    # reset_num_timesteps=False → le compteur continue depuis la dernière fois
    model.learn(
        total_timesteps=TIMESTEPS,
        reset_num_timesteps=False,
        tb_log_name="PPO"  # nom du run dans TensorBoard
    )
    
    # Le nom du fichier = nombre total de steps (pratique pour retrouver)
    save_path = f"{models_dir}/{TIMESTEPS * i}"
    model.save(save_path)
    print(f"Iteration {i} — Sauvegardé : {save_path}.zip")

env.close()
print("\nEntraînement terminé !")

### Ce que `reset_num_timesteps=False` fait

Sans ce paramètre, chaque appel à `.learn()` repart de 0 dans les logs.
Avec `False`, le compteur continue : les graphes TensorBoard sont continus.

## 3. Vérifier les fichiers sauvegardés

In [ ]:
saved_models = sorted(os.listdir(models_dir))
print("Modèles sauvegardés :")
for f in saved_models:
    path = os.path.join(models_dir, f)
    size_kb = os.path.getsize(path) / 1024
    print(f"  {f}  ({size_kb:.1f} KB)")

## 4. Charger un modèle sauvegardé

On peut charger n'importe quel checkpoint et l'évaluer ou continuer l'entraînement.

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy

env = gym.make('LunarLander-v3')

# Charger un checkpoint (SB3 ajoute .zip automatiquement)
model_path = f"{models_dir}/{TIMESTEPS * N_ITERATIONS}"
loaded_model = PPO.load(model_path, env=env)

print(f"Modèle chargé depuis : {model_path}.zip")

mean_reward, std_reward = evaluate_policy(loaded_model, env, n_eval_episodes=10)
print(f"Reward moyen : {mean_reward:.2f} ± {std_reward:.2f}")

env.close()

## 5. Comparer plusieurs checkpoints

Intérêt des checkpoints : voir si plus de steps = meilleures performances.

In [ ]:
env = gym.make('LunarLander-v3')

results = []
for i in range(1, N_ITERATIONS + 1):
    path = f"{models_dir}/{TIMESTEPS * i}"
    m = PPO.load(path, env=env)
    mean, std = evaluate_policy(m, env, n_eval_episodes=5)
    results.append((TIMESTEPS * i, mean, std))
    print(f"{TIMESTEPS * i:>8} steps → reward moyen : {mean:>8.2f} ± {std:.2f}")

env.close()

## 6. TensorBoard

TensorBoard affiche les courbes d'entraînement (reward, loss, etc.) de façon interactive.

### Lancer TensorBoard

Dans un terminal (pas dans le notebook) :

```bash
tensorboard --logdir=logs
```

Puis ouvrir http://localhost:6006 dans le navigateur.

### Métriques importantes

| Métrique | Signification |
|----------|---------------|
| `rollout/ep_rew_mean` | Récompense moyenne par épisode (à maximiser) |
| `rollout/ep_len_mean` | Longueur moyenne des épisodes |
| `train/value_loss` | Erreur du réseau de valeur (doit décroître) |
| `train/entropy_loss` | Entropie de la politique (exploration) |
| `train/approx_kl` | Divergence KL (mesure le changement de politique) |

In [ ]:
# Optionnel : lancer TensorBoard directement depuis le notebook
%load_ext tensorboard
%tensorboard --logdir logs

## 7. Comparer PPO vs A2C dans TensorBoard

In [ ]:
from stable_baselines3 import A2C

# Entraîner A2C avec logs séparés pour comparer
models_dir_a2c = "models/A2C"
os.makedirs(models_dir_a2c, exist_ok=True)

env = gym.make('LunarLander-v3')
model_a2c = A2C('MlpPolicy', env, verbose=0, tensorboard_log=logs_dir)

for i in range(1, N_ITERATIONS + 1):
    model_a2c.learn(
        total_timesteps=TIMESTEPS,
        reset_num_timesteps=False,
        tb_log_name="A2C"  # run séparé dans TensorBoard
    )
    model_a2c.save(f"{models_dir_a2c}/{TIMESTEPS * i}")

env.close()
print("Entraînement A2C terminé — comparer PPO vs A2C dans TensorBoard")

## Résumé

| Action | Code |
|--------|------|
| Sauvegarder | `model.save("chemin")` |
| Charger | `PPO.load("chemin", env=env)` |
| Logs TensorBoard | `PPO(..., tensorboard_log="logs")` |
| Continuer l'entraînement | `model.learn(..., reset_num_timesteps=False)` |

**Prochaine étape** : créer son propre environnement → notebook `03_custom_environment.ipynb`